
<div style="text-align: right;">
  <a href="../../Files/ask_csv_proxy.zip"
     download="ask_csv_proxy.zip"
     class="md-button md-button--primary">
    ⬇️ Download Notebook
  </a>


## Install and import utility.ipynb

In [1]:
%run ./utility.ipynb

In [ ]:
%pip install -e ./utility.ipynb
%pip install import_ipynb

In [3]:
import import_ipynb
import utility

## Authenticate and get token

In [ ]:
token = utility.get_oauth_token()

## Build prompt

In [5]:

questions = """
            1. Which manufacturer has the highest average resale value?
            2. What is the relationship between engine size and fuel efficiency?
            3. Which vehicle type has the highest average horsepower?
            4. How does price correlate with power performance factor?
            5. Which models had the best sales performance relative to fuel efficiency?
            6. What are the top 3 manufacturers by total sales?
            7. Do larger wheelbases correlate with higher resale values?"""

prompt = """You are a data analysis assistant. 
                1. The user provides a CSV file and asks questions about it.
                2. Profile the data (columns, data types), perform any required aggregations or filters, and answer clearly with supporting evidence such as which columns were used, what assumptions were made, and a short reasoning summary.
                3. Return a concise, factual answer and no extra explanations.
                 4. Answer all the questions in the same order. 
                 
            The Questions are: {questions}""".format(questions=questions)

## Build request body

In [6]:
base64encoded = utility.base64_encode_file(pdf_path='D:/Users/572981/Downloads/example_usecase_refactored/Car_sales.csv')
inferenceConfig = {
            "maxTokens": 512,
            "temperature": 0.5,
            "topP": 0.9
        }
body = {
        "inferenceConfig": inferenceConfig,
        "messages": [
            {   "role": "user",
                "content": [
                    {"text": prompt},
                    {"document": {
                            "format": "csv",
                            "name":'car_sales',
                            "source": {
                                "bytes": base64encoded }}
                    }]
            }]
        }
request_body={
    "methodName" :  "converse",
    "parameters" : {
        "body" : body
    }
}

## Invoke Modelops

In [7]:
modelId = "anthropic.claude-3-5-sonnet-20241022-v2:0"
jobId = utility.invoke_llm_proxy(model_id=modelId, token=token, request_body=request_body)

Poll job status via Get Job by job ID until succeeded/failed and Get Data is Successfull

In [ ]:
if utility.wait_for_job_completion(jobId, token):
    modelops_resp = utility.get_response(jobId, token)
    response={}
    response["body"] = json.dumps(modelops_resp["output"]['message']['content'][0]['text'])
else:
    response = {}

print(response)